### Импорты

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
)
import mlflow

### Собираем датасет

In [3]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
val_df = pd.read_csv("valid.csv")
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(1537, 8)
(330, 8)
(329, 8)


In [4]:
train_df.dtypes

text                       object
id                          int64
product_quality              bool
staff_service                bool
queues_checkout              bool
assortment_availability      bool
store_cleanliness            bool
other_problem                bool
dtype: object

In [5]:
labels = [
    "product_quality",
    "staff_service",
    "queues_checkout",
    "assortment_availability",
    "store_cleanliness",
    "other_problem",
]

In [6]:
for part in [train_df, val_df, test_df]:
    part[labels] = part[labels].astype(float)
    part["labels"] = part[labels].values.tolist()

In [7]:
train_df.head(3)

,text,id,product_quality,staff_service,queues_checkout,assortment_availability,store_cleanliness,other_problem,labels
0,Не самый хороший Гуливер в городе. Неудобное в...,361967,0.0,0.0,0.0,1.0,0.0,0.0,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]"
1,Заходили с ребенком купили макаронс цены на не...,34803,1.0,0.0,0.0,0.0,0.0,0.0,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
2,"Отстойно, цены высокие, всегда грязь на полу, ...",235565,0.0,1.0,1.0,1.0,1.0,0.0,"[0.0, 1.0, 1.0, 1.0, 1.0, 0.0]"


In [8]:
dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(
            train_df[["text", "labels"]], preserve_index=False
        ),
        "validation": Dataset.from_pandas(
            val_df[["text", "labels"]], preserve_index=False
        ),
        "test": Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False),
    }
)

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 1537
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 329
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 330
    })
})

### Tokenize

In [ ]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=350,
    )


tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1537 [00:00<?, ? examples/s]

Map:   0%|          | 0/329 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

In [10]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1537
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 329
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 330
    })
})

### Прогон 1

In [11]:
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= 0.5).astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "micro_precision": precision_score(
            labels, preds, average="micro", zero_division=0
        ),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "subset_accuracy": accuracy_score(labels, preds),
    }

In [13]:
print(torch.backends.mps.is_available())

True


In [ ]:
training_args = TrainingArguments(
    output_dir="models/rubert_1",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [15]:
train_result = trainer.train()

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Weighted F1,Micro Precision,Micro Recall,Subset Accuracy
1,0.588949,0.516839,0.431762,0.115308,0.252373,0.528875,0.364780,0.203647
2,0.499737,0.494238,0.433375,0.116000,0.253887,0.533742,0.364780,0.203647
3,0.486775,0.482254,0.460274,0.131148,0.287040,0.664032,0.352201,0.221884
4,0.471522,0.469913,0.484150,0.158753,0.332583,0.774194,0.352201,0.227964
5,0.467789,0.465176,0.500000,0.179936,0.361889,0.794521,0.364780,0.243161


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
pred_output = trainer.predict(tokenized_dataset["test"])
logits = pred_output.predictions
true_labels = pred_output.label_ids
probs = sigmoid(logits)
pred_labels = (probs >= 0.5).astype(int)
print(
    classification_report(
        true_labels, pred_labels, target_names=labels, zero_division=0
    )
)

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


                         precision    recall  f1-score   support

        product_quality       0.67      0.07      0.13       109
          staff_service       0.78      0.91      0.84       179
        queues_checkout       0.00      0.00      0.00        61
assortment_availability       0.00      0.00      0.00        34
      store_cleanliness       0.00      0.00      0.00        42
          other_problem       0.00      0.00      0.00        44

              micro avg       0.77      0.36      0.49       469
              macro avg       0.24      0.16      0.16       469
           weighted avg       0.45      0.36      0.35       469
            samples avg       0.52      0.39      0.43       469



In [17]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
for threshold in thresholds:
    pred_labels_thr = (probs >= threshold).astype(int)

    print(
        f"threshold={threshold:.2f}",
        "micro_f1:",
        round(
            f1_score(true_labels, pred_labels_thr, average="micro", zero_division=0), 4
        ),
        "macro_f1:",
        round(
            f1_score(true_labels, pred_labels_thr, average="macro", zero_division=0), 4
        ),
        "weighted_f1:",
        round(
            f1_score(true_labels, pred_labels_thr, average="weighted", zero_division=0),
            4,
        ),
    )

threshold=0.10 micro_f1: 0.383 macro_f1: 0.36 weighted_f1: 0.4803
threshold=0.20 micro_f1: 0.4867 macro_f1: 0.3655 weighted_f1: 0.4826
threshold=0.30 micro_f1: 0.5502 macro_f1: 0.2282 weighted_f1: 0.414
threshold=0.40 micro_f1: 0.5629 macro_f1: 0.2276 weighted_f1: 0.4329
threshold=0.50 micro_f1: 0.4935 macro_f1: 0.1616 weighted_f1: 0.3503
threshold=0.60 micro_f1: 0.3921 macro_f1: 0.1251 weighted_f1: 0.2865
threshold=0.70 micro_f1: 0.1046 macro_f1: 0.0419 weighted_f1: 0.0959


In [18]:
best_threshold = 0.2
pred_labels_best = (probs >= best_threshold).astype(int)
print(
    classification_report(
        true_labels, pred_labels_best, target_names=labels, zero_division=0
    )
)

                         precision    recall  f1-score   support

        product_quality       0.33      1.00      0.50       109
          staff_service       0.54      1.00      0.70       179
        queues_checkout       0.20      0.95      0.34        61
assortment_availability       0.21      0.32      0.25        34
      store_cleanliness       0.25      0.31      0.27        42
          other_problem       0.24      0.09      0.13        44

              micro avg       0.35      0.80      0.49       469
              macro avg       0.29      0.61      0.37       469
           weighted avg       0.37      0.80      0.48       469
            samples avg       0.37      0.78      0.48       469



### Прогон 2

https://discuss.huggingface.co/t/multilabel-classification-performance-metrics-using-trainer-api/11911

In [38]:
label_counts = train_df[labels].sum().values.astype(float)
total_count = len(train_df)
neg_counts = total_count - label_counts
pos_weight_values = neg_counts / label_counts
pos_weight = torch.tensor(pos_weight_values, dtype=torch.float32)
print(pos_weight)

tensor([1.9110, 0.8168, 4.2457, 5.9234, 6.5343, 8.6062])


In [39]:
class WeightedMultilabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(logits.device))
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [46]:
training_args = TrainingArguments(
    output_dir="models/rubert_2",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    num_train_epochs=8,
    weight_decay=0.01,
    save_total_limit=2,
    report_to="none",
    seed=42,
)

In [47]:
trainer = WeightedMultilabelTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [48]:
train_result = trainer.train()

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Weighted F1,Micro Precision,Micro Recall,Subset Accuracy
1,0.332369,0.656022,0.738351,0.682396,0.747145,0.644757,0.863732,0.422492
2,0.302718,0.656001,0.749311,0.693212,0.754501,0.666667,0.855346,0.431611
3,0.283934,0.673246,0.751640,0.695897,0.751976,0.679661,0.840671,0.440729
4,0.263471,0.653098,0.754613,0.696441,0.759094,0.673806,0.857442,0.443769
5,0.255871,0.668570,0.751165,0.695196,0.754238,0.676174,0.844864,0.431611
6,0.248985,0.655886,0.761546,0.705166,0.766358,0.691781,0.846960,0.455927
7,0.240637,0.660181,0.760190,0.704137,0.764056,0.693772,0.840671,0.455927
8,0.241888,0.656605,0.754717,0.699633,0.759339,0.686106,0.838574,0.446809


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [49]:
pred_output = trainer.predict(tokenized_dataset["test"])
logits = pred_output.predictions
true_labels = pred_output.label_ids
probs = sigmoid(logits)
pred_labels = (probs >= 0.5).astype(int)
print(
    classification_report(
        true_labels, pred_labels, target_names=labels, zero_division=0
    )
)

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


                         precision    recall  f1-score   support

        product_quality       0.74      0.87      0.80       109
          staff_service       0.79      0.92      0.85       179
        queues_checkout       0.71      0.75      0.73        61
assortment_availability       0.52      0.76      0.62        34
      store_cleanliness       0.60      0.86      0.71        42
          other_problem       0.46      0.61      0.52        44

              micro avg       0.69      0.84      0.76       469
              macro avg       0.64      0.80      0.71       469
           weighted avg       0.70      0.84      0.76       469
            samples avg       0.73      0.83      0.75       469



In [44]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
for threshold in thresholds:
    pred_labels_thr = (probs >= threshold).astype(int)

    print(
        f"threshold={threshold:.2f}",
        "micro_f1:",
        round(
            f1_score(true_labels, pred_labels_thr, average="micro", zero_division=0), 4
        ),
        "macro_f1:",
        round(
            f1_score(true_labels, pred_labels_thr, average="macro", zero_division=0), 4
        ),
        "weighted_f1:",
        round(
            f1_score(true_labels, pred_labels_thr, average="weighted", zero_division=0),
            4,
        ),
    )

threshold=0.10 micro_f1: 0.4814 macro_f1: 0.4363 weighted_f1: 0.5356
threshold=0.20 micro_f1: 0.5907 macro_f1: 0.5352 weighted_f1: 0.6233
threshold=0.30 micro_f1: 0.6535 macro_f1: 0.5972 weighted_f1: 0.6762
threshold=0.40 micro_f1: 0.6895 macro_f1: 0.6258 weighted_f1: 0.7067
threshold=0.50 micro_f1: 0.7273 macro_f1: 0.6666 weighted_f1: 0.7414
threshold=0.60 micro_f1: 0.7538 macro_f1: 0.6976 weighted_f1: 0.7614
threshold=0.70 micro_f1: 0.745 macro_f1: 0.6857 weighted_f1: 0.7462
threshold=0.80 micro_f1: 0.6974 macro_f1: 0.6399 weighted_f1: 0.6957
threshold=0.90 micro_f1: 0.5465 macro_f1: 0.5026 weighted_f1: 0.5414


In [45]:
pred_output = trainer.predict(tokenized_dataset["test"])
logits = pred_output.predictions
true_labels = pred_output.label_ids
probs = sigmoid(logits)
pred_labels = (probs >= 0.6).astype(int)
print(
    classification_report(
        true_labels, pred_labels, target_names=labels, zero_division=0
    )
)

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


                         precision    recall  f1-score   support

        product_quality       0.75      0.82      0.78       109
          staff_service       0.85      0.87      0.86       179
        queues_checkout       0.78      0.74      0.76        61
assortment_availability       0.44      0.71      0.54        34
      store_cleanliness       0.63      0.86      0.73        42
          other_problem       0.46      0.59      0.52        44

              micro avg       0.71      0.80      0.75       469
              macro avg       0.65      0.76      0.70       469
           weighted avg       0.73      0.80      0.76       469
            samples avg       0.72      0.79      0.73       469



### Прогон 3

In [52]:
training_arg_3 = TrainingArguments(
    output_dir="models/rubert_3",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    num_train_epochs=15,
    weight_decay=0.01,
    report_to="none",
    seed=42,
)

trainer3 = WeightedMultilabelTrainer(
    model=model,
    args=training_arg_3,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [53]:
train_result3 = trainer3.train()

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Weighted F1,Micro Precision,Micro Recall,Subset Accuracy
1,0.217968,0.681615,0.750469,0.702701,0.754978,0.679117,0.838574,0.440729
2,0.190695,0.663208,0.764088,0.707887,0.767626,0.701754,0.838574,0.477204
3,0.169619,0.699265,0.773385,0.719347,0.774453,0.716071,0.840671,0.498480
4,0.151092,0.710594,0.771014,0.710298,0.772335,0.715054,0.836478,0.504559
5,0.139278,0.726064,0.781065,0.723010,0.781633,0.737430,0.830189,0.516717
6,0.131062,0.732219,0.773492,0.709857,0.772727,0.732210,0.819706,0.510638
7,0.121418,0.750333,0.781219,0.719339,0.780138,0.746183,0.819706,0.522796
8,0.116425,0.725378,0.760396,0.704015,0.763957,0.720450,0.805031,0.501520
9,0.109870,0.756851,0.780100,0.719114,0.779855,0.742424,0.821803,0.522796
10,0.110209,0.748999,0.776542,0.717283,0.777152,0.750000,0.805031,0.525836


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [54]:
pred_output = trainer3.predict(tokenized_dataset["test"])
logits = pred_output.predictions
true_labels = pred_output.label_ids
probs = sigmoid(logits)
pred_labels = (probs >= 0.5).astype(int)
print(
    classification_report(
        true_labels, pred_labels, target_names=labels, zero_division=0
    )
)

/Users/admin/Documents/smadimo_gp_5_deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


                         precision    recall  f1-score   support

        product_quality       0.77      0.81      0.79       109
          staff_service       0.81      0.93      0.87       179
        queues_checkout       0.72      0.72      0.72        61
assortment_availability       0.57      0.62      0.59        34
      store_cleanliness       0.69      0.79      0.73        42
          other_problem       0.61      0.52      0.56        44

              micro avg       0.75      0.80      0.77       469
              macro avg       0.69      0.73      0.71       469
           weighted avg       0.74      0.80      0.77       469
            samples avg       0.74      0.79      0.74       469

